# LeetCode 3836 — Maximum Score Using Exactly K Pairs bằng Python

- Link: https://leetcode.com/problems/maximum-score-using-exactly-k-pairs/
- Độ khó: **Hard**
- Chủ đề: Quy hoạch động hai dãy (kiểu LCS) có thêm chiều "số cặp", xử lý giá trị âm với `−∞`, tối ưu bộ nhớ theo tầng, max tiền tố 2D

Skeleton trên LeetCode (Python3):

```python
class Solution:
    def maxScore(self, nums1: List[int], nums2: List[int], k: int) -> int:
```

## 1. Mục tiêu học

1. Nhận ra cấu trúc "ghép cặp **tăng dần** chỉ số trên hai dãy" giống *Longest Common Subsequence* / *Max Dot Product of Two Subsequences* (LC 1458).
2. Thêm chiều **số cặp đã chọn** vào trạng thái và dùng **`−∞`** cho trạng thái không đạt được (vì điểm có thể âm, **bắt buộc** đúng `k` cặp).
3. Viết DP 3 chiều `f[i][j][t]` với ba chuyển trạng thái: bỏ phần tử của `nums1`, bỏ phần tử của `nums2`, hoặc ghép cặp.
4. Giảm bộ nhớ bằng cách duyệt **theo tầng `t`** (chỉ giữ hai tầng) và nhìn bài theo góc "max tiền tố 2D".
5. Kiểm thử với vét cạn tổ hợp.

## 2. Đề bài & ví dụ

**Phát biểu lại.** Cho `nums1` (độ dài `n`), `nums2` (độ dài `m`) và `k`. Chọn **đúng `k`** cặp chỉ số `(i₁, j₁), …, (i_k, j_k)` với `i₁ < … < i_k` và `j₁ < … < j_k`. Mỗi cặp được `nums1[i] · nums2[j]` điểm. Tìm tổng điểm lớn nhất.

**Ví dụ 1:** `nums1 = [1,3,2], nums2 = [4,5,1], k = 2` → `22` (`3·4 + 2·5`).

**Ví dụ 2:** `nums1 = [-2,0,5], nums2 = [-3,4,-1,2], k = 2` → `26` (`(−2)(−3) + 5·4`).

**Ví dụ 3:** `nums1 = [-3,-2], nums2 = [1,2], k = 2` → `−7` (bắt buộc chọn đủ 2 cặp dù điểm âm).

**Ràng buộc:** `1 ≤ n, m ≤ 100`, `|nums[i]| ≤ 10^6`, `1 ≤ k ≤ min(n, m)`.

In [1]:
from typing import List
from itertools import combinations
from functools import cache
import random, time

NEG = float('-inf')
examples = [([1, 3, 2], [4, 5, 1], 2, 22), ([-2, 0, 5], [-3, 4, -1, 2], 2, 26), ([-3, -2], [1, 2], 2, -7)]
for a, b, k, exp in examples:
    print(a, b, "k =", k, "-> kỳ vọng", exp)

[1, 3, 2] [4, 5, 1] k = 2 -> kỳ vọng 22
[-2, 0, 5] [-3, 4, -1, 2] k = 2 -> kỳ vọng 26
[-3, -2] [1, 2] k = 2 -> kỳ vọng -7


## 3. Kiến thức nền

### 3.1. DP trên hai tiền tố (kiểu LCS)

Trạng thái `f[i][j]` mô tả lời giải tốt nhất khi chỉ dùng `nums1[0..i-1]` và `nums2[0..j-1]`. Chuyển trạng thái nhìn vào **phần tử cuối** của hai tiền tố: bỏ `nums1[i-1]`, bỏ `nums2[j-1]`, hoặc dùng cả hai.

In [2]:
def lcs(a, b):
    n, m = len(a), len(b)
    f = [[0] * (m + 1) for _ in range(n + 1)]
    for i in range(1, n + 1):
        for j in range(1, m + 1):
            f[i][j] = f[i - 1][j - 1] + 1 if a[i - 1] == b[j - 1] else max(f[i - 1][j], f[i][j - 1])
    return f[n][m]

print(lcs("abcde", "ace"))

3


### 3.2. `−∞` cho trạng thái không hợp lệ

Vì điểm có thể âm và phải chọn **đúng** `k` cặp, ta không được khởi tạo bằng `0` (sẽ hiểu nhầm là "chọn ít cặp hơn"). Dùng `float('-inf')`: cộng với số hữu hạn vẫn là `−∞`, và `max(−∞, x) = x`.

In [3]:
print(NEG + 5, max(NEG, -10**12), NEG < -10**18)

-inf -1000000000000 True


### 3.3. `itertools.combinations` — chọn chỉ số tăng dần

`combinations(range(n), k)` sinh mọi bộ `k` chỉ số tăng dần — đúng dạng `i₁ < … < i_k`. Ghép bộ thứ nhất với bộ thứ hai theo thứ tự cho một cách chọn cặp hợp lệ.

In [4]:
print(list(combinations(range(4), 2)))

[(0, 1), (0, 2), (0, 3), (1, 2), (1, 3), (2, 3)]


### 3.4. Max tiền tố 2D

`M[i][j] = max_{i' ≤ i, j' ≤ j} g[i'][j']` tính bằng `M[i][j] = max(g[i][j], M[i-1][j], M[i][j-1])`. Dùng để trả lời "giá trị tốt nhất của một cặp **trước** `(i, j)`" trong `O(1)`.

In [5]:
g = [[3, -1, 2], [0, 5, -4], [1, 1, 1]]
M = [[NEG] * 3 for _ in range(3)]
for i in range(3):
    for j in range(3):
        M[i][j] = max(g[i][j], M[i - 1][j] if i else NEG, M[i][j - 1] if j else NEG)
print(M)

[[3, 3, 3], [3, 5, 5], [3, 5, 5]]


## 4. Ý tưởng nền tảng / cách nghĩ trực giác

Ví dụ 2: `nums1 = [-2, 0, 5]`, `nums2 = [-3, 4, -1, 2]`, `k = 2`.

- Cặp `(−2, −3)` cho `+6` (âm × âm = dương).
- Cặp `(5, 4)` cho `20`. Chỉ số tăng trên cả hai dãy: `(0, 0)` rồi `(2, 1)` ✔.
- Tổng `26`.

Tham lam "ghép hai số lớn nhất" thất bại vì ràng buộc **thứ tự** và vì **âm × âm** có thể rất lớn. Ta cần xét mọi cách — nhưng gộp theo trạng thái `(đã xét i phần tử của nums1, j phần tử của nums2, đã chọn t cặp)`.

Ví dụ 3 cho thấy phải chọn **đủ** `k` cặp kể cả khi mọi tích đều âm: `(−3)(1) + (−2)(2) = −7` (cách duy nhất).

## 5. Cách 1 — Brute Force: thử mọi bộ chỉ số

Chọn `k` chỉ số tăng dần của `nums1` và `k` chỉ số tăng dần của `nums2`, ghép theo thứ tự. Số cách `C(n,k) · C(m,k)` — chỉ dùng khi nhỏ.

In [6]:
def brute(a, b, k):
    best = NEG
    for I in combinations(range(len(a)), k):
        for J in combinations(range(len(b)), k):
            best = max(best, sum(a[i] * b[j] for i, j in zip(I, J)))
    return best

for a, b, k, exp in examples:
    print(brute(a, b, k), "kỳ vọng", exp)

22 kỳ vọng 22
26 kỳ vọng 26
-7 kỳ vọng -7


**Nút thắt:** `C(100, 50)² ≈ 10^{58}`. Nhưng nhiều cách chọn có **chung tiền tố quyết định** ⇒ DP.

## 6. Ý tưởng thuật toán / bước đột phá

**Trạng thái:** `f[i][j][t]` = điểm lớn nhất khi chọn **đúng** `t` cặp, chỉ dùng `nums1[0..i-1]` và `nums2[0..j-1]`; `−∞` nếu không thể.

**Chuyển trạng thái** (xét phần tử cuối):

- bỏ `nums1[i-1]`: `f[i-1][j][t]`;
- bỏ `nums2[j-1]`: `f[i][j-1][t]`;
- ghép `(i-1, j-1)`: `f[i-1][j-1][t-1] + nums1[i-1] · nums2[j-1]`.

`f[i][j][t]` = max ba giá trị. Khởi tạo `f[i][j][0] = 0` với mọi `i, j` (không chọn cặp nào thì điểm 0), còn lại `−∞`. Đáp án `f[n][m][k]`.

**Quan sát giảm bộ nhớ:** tầng `t` chỉ phụ thuộc tầng `t` (hai chuyển "bỏ") và tầng `t−1` (chuyển "ghép") ⇒ duyệt `t = 1..k`, giữ hai bảng `(n+1)×(m+1)`.

## 7. Các cách tiếp theo

- **Cách 2:** đệ quy có nhớ `dfs(i, j, t)` (từ trên xuống).
- **Cách 3:** DP từ dưới lên theo **tầng `t`** với hai bảng 2D (lời giải nộp chính).
- **Cách 4:** góc nhìn "cặp cuối cùng": `h_t[i][j]` = điểm tốt nhất với cặp thứ `t` **đúng là** `(i, j)`, tính bằng **max tiền tố 2D** của `h_{t−1}`.

### 7.1. Cách 2 — Đệ quy có nhớ

In [7]:
def memo(a, b, k):
    n, m = len(a), len(b)

    @cache
    def dfs(i, j, t):           # dùng a[i:], b[j:], cần chọn thêm t cặp
        if t == 0:
            return 0
        if n - i < t or m - j < t:
            return NEG           # không đủ phần tử
        return max(dfs(i + 1, j, t), dfs(i, j + 1, t), a[i] * b[j] + dfs(i + 1, j + 1, t - 1))

    res = dfs(0, 0, k)
    dfs.cache_clear()
    return res

for a, b, k, exp in examples:
    print(memo(a, b, k), "kỳ vọng", exp)

22 kỳ vọng 22
26 kỳ vọng 26
-7 kỳ vọng -7


**Độ phức tạp:** `O(n · m · k)` trạng thái, mỗi trạng thái `O(1)` ⇒ tới `10^6` lời gọi; độ sâu đệ quy ≤ `n + m` ≈ 200 (an toàn). Bộ nhớ cache `O(n m k)`.

### 7.2. Cách 3 — DP theo tầng `t` (lời giải chính)

**Giải thích từng đoạn:**

1. `prev[i][j]` = tầng `t − 1`; ban đầu (tầng 0) mọi ô bằng `0`.
2. Tầng `t`: `cur[i][j] = max(cur[i-1][j], cur[i][j-1], prev[i-1][j-1] + a[i-1]·b[j-1])`, với `cur[0][·] = cur[·][0] = −∞` (không thể có `t ≥ 1` cặp từ tiền tố rỗng).
3. Sau `k` tầng, đáp án `prev[n][m]`.

In [8]:
def layered(a, b, k):
    n, m = len(a), len(b)
    prev = [[0] * (m + 1) for _ in range(n + 1)]
    for t in range(1, k + 1):
        cur = [[NEG] * (m + 1) for _ in range(n + 1)]
        for i in range(1, n + 1):
            ai = a[i - 1]; row, up, pr = cur[i], cur[i - 1], prev[i - 1]
            for j in range(1, m + 1):
                best = up[j]
                if row[j - 1] > best: best = row[j - 1]
                cand = pr[j - 1] + ai * b[j - 1]
                if cand > best: best = cand
                row[j] = best
        prev = cur
    return prev[n][m]

for a, b, k, exp in examples:
    print(layered(a, b, k), "kỳ vọng", exp)

22 kỳ vọng 22
26 kỳ vọng 26
-7 kỳ vọng -7


**Độ phức tạp Cách 3:** `O(k · n · m)` ≤ `10^6` phép tính, bộ nhớ `O(n · m)`.

### 7.3. Cách 4 — Cặp cuối cố định + max tiền tố 2D

**Định nghĩa:** `h_t[i][j]` = điểm tốt nhất của `t` cặp trong đó cặp **cuối** là `(i, j)` (0-indexed). Khi đó

`h_t[i][j] = a[i]·b[j] + max_{i' < i, j' < j} h_{t−1}[i'][j']` (với `t = 1` thì phần max là `0`).

Phần max là max tiền tố 2D của `h_{t−1}` tại `(i−1, j−1)`. Đáp án `max_{i, j} h_k[i][j]`.

In [9]:
def last_pair(a, b, k):
    n, m = len(a), len(b)
    h = [[a[i] * b[j] for j in range(m)] for i in range(n)]          # t = 1
    for t in range(2, k + 1):
        M = [[NEG] * m for _ in range(n)]                              # max tiền tố 2D của h
        for i in range(n):
            for j in range(m):
                M[i][j] = max(h[i][j], M[i - 1][j] if i else NEG, M[i][j - 1] if j else NEG)
        h = [[a[i] * b[j] + (M[i - 1][j - 1] if i and j else NEG) for j in range(m)] for i in range(n)]
    return max(max(row) for row in h)

for a, b, k, exp in examples:
    print(last_pair(a, b, k), "kỳ vọng", exp)

22 kỳ vọng 22
26 kỳ vọng 26
-7 kỳ vọng -7


In [10]:
class SolutionLastPair:
    def maxScore(self, nums1: List[int], nums2: List[int], k: int) -> int:
        return last_pair(nums1, nums2, k)

print([SolutionLastPair().maxScore(a, b, k) for a, b, k, _ in examples])

[22, 26, -7]


**Độ phức tạp Cách 4:** `O(k · n · m)` thời gian, `O(n · m)` bộ nhớ. Góc nhìn này hữu ích khi muốn biến thể "khoảng cách giữa hai cặp liên tiếp" (thay max tiền tố bằng max trên vùng khác).

## 8. TRACE từng bước

Trace Cách 3 trên ví dụ 1 (`a = [1,3,2]`, `b = [4,5,1]`, `k = 2`). In bảng `cur` sau mỗi tầng (hàng `i = 0..3`, cột `j = 0..3`).

In [11]:
def trace(a, b, k):
    n, m = len(a), len(b)
    prev = [[0] * (m + 1) for _ in range(n + 1)]
    fmt = lambda v: " -∞" if v == NEG else f"{v:3}"
    for t in range(1, k + 1):
        cur = [[NEG] * (m + 1) for _ in range(n + 1)]
        for i in range(1, n + 1):
            for j in range(1, m + 1):
                cur[i][j] = max(cur[i - 1][j], cur[i][j - 1], prev[i - 1][j - 1] + a[i - 1] * b[j - 1])
        print(f"tầng t = {t}:")
        for i in range(n + 1):
            print("   ", " ".join(fmt(v) for v in cur[i]))
        prev = cur
    return prev[n][m]

print("đáp án:", trace([1, 3, 2], [4, 5, 1], 2))

tầng t = 1:
     -∞  -∞  -∞  -∞
     -∞   4   5   5
     -∞  12  15  15
     -∞  12  15  15
tầng t = 2:
     -∞  -∞  -∞  -∞
     -∞  -∞  -∞  -∞
     -∞  -∞  19  19
     -∞  -∞  22  22
đáp án: 22


Đọc bảng tầng 1: `cur[3][3] = 15` là tích lớn nhất `3·5`. Tầng 2: `cur[3][2] = 22` = `cur_tầng1[2][1] (= 3·4 = 12) + 2·5`. Lưu ý `cur[1][·]` ở tầng 2 là `−∞` vì một phần tử của `nums1` không tạo đủ 2 cặp.

In [12]:
# Truy vết các cặp được chọn (dùng bảng 3D đầy đủ)
def reconstruct(a, b, k):
    n, m = len(a), len(b)
    f = [[[NEG] * (k + 1) for _ in range(m + 1)] for _ in range(n + 1)]
    for i in range(n + 1):
        for j in range(m + 1):
            f[i][j][0] = 0
    for i in range(1, n + 1):
        for j in range(1, m + 1):
            for t in range(1, k + 1):
                f[i][j][t] = max(f[i - 1][j][t], f[i][j - 1][t], f[i - 1][j - 1][t - 1] + a[i - 1] * b[j - 1])
    pairs, i, j, t = [], n, m, k
    while t:
        if f[i][j][t] == f[i - 1][j][t]: i -= 1
        elif f[i][j][t] == f[i][j - 1][t]: j -= 1
        else:
            pairs.append((i - 1, j - 1)); i -= 1; j -= 1; t -= 1
    return f[n][m][k], pairs[::-1]

for a, b, k, _ in examples:
    print(reconstruct(a, b, k))

(22, [(1, 0), (2, 1)])
(26, [(0, 0), (2, 1)])
(-7, [(0, 0), (1, 1)])


## 9. Code nộp LeetCode

In [13]:
class Solution:
    def maxScore(self, nums1: List[int], nums2: List[int], k: int) -> int:
        NEG = float('-inf')
        n, m = len(nums1), len(nums2)
        prev = [[0] * (m + 1) for _ in range(n + 1)]       # tầng t = 0
        for _ in range(k):
            cur = [[NEG] * (m + 1) for _ in range(n + 1)]
            for i in range(1, n + 1):
                ai = nums1[i - 1]
                row, up, pr = cur[i], cur[i - 1], prev[i - 1]
                for j in range(1, m + 1):
                    best = up[j]                             # bỏ nums1[i-1]
                    if row[j - 1] > best:                    # bỏ nums2[j-1]
                        best = row[j - 1]
                    cand = pr[j - 1] + ai * nums2[j - 1]     # ghép cặp
                    if cand > best:
                        best = cand
                    row[j] = best
            prev = cur
        return prev[n][m]

sol = Solution()
for a, b, k, exp in examples:
    got = sol.maxScore(a, b, k)
    print(got, "✔" if got == exp else "✘")

22 ✔
26 ✔
-7 ✔


## 10. Vì sao thuật toán đúng

1. **Phân loại theo phần tử cuối:** xét mọi cách chọn `t` cặp trong `nums1[0..i-1] × nums2[0..j-1]`. Hoặc `nums1[i-1]` không được dùng (giảm `i`), hoặc `nums2[j-1]` không được dùng (giảm `j`), hoặc cả hai được dùng — khi đó chúng **phải ghép với nhau** (vì chúng là phần tử lớn nhất trên mỗi dãy, cặp chứa chúng là cặp cuối của cả hai thứ tự). Ba trường hợp phủ mọi cách chọn.
2. **Cấu trúc con tối ưu:** trong mỗi trường hợp, phần còn lại là một bài toán con cùng dạng, độc lập.
3. **`−∞`:** trạng thái không đạt được (thiếu phần tử) không bao giờ thắng phép `max` trừ khi mọi lựa chọn đều không đạt — đúng nghĩa "không thể". Do `k ≤ min(n, m)`, `f[n][m][k]` luôn hữu hạn.
4. **Tầng:** tầng `t` chỉ cần tầng `t − 1` ⇒ giữ hai bảng không làm mất thông tin.

## 11. Phân tích Time / Space Complexity

| Cách | Time | Space |
|---|---|---|
| 1. Vét cạn | `O(C(n,k) C(m,k) k)` | `O(k)` |
| 2. Memo | `O(n m k)` | `O(n m k)` |
| 3. DP theo tầng | `O(n m k)` | `O(n m)` |
| 4. Cặp cuối + max tiền tố | `O(n m k)` | `O(n m)` |

Với `n = m = k = 100`: `10^6` ô — khoảng 1 giây trong Python thuần.

## 12. Bộ test

In [14]:
def run_all(a, b, k, with_brute=True):
    res = [memo(a, b, k), layered(a, b, k), last_pair(a, b, k), Solution().maxScore(a, b, k)]
    if with_brute:
        res.append(brute(a, b, k))
    return res

for a, b, k, exp in examples:
    assert all(v == exp for v in run_all(a, b, k))
print("Ví dụ đề: OK")

Ví dụ đề: OK


In [15]:
# 12.2 ca biên (tự dựng, kiểm chứng bằng vét cạn)
cases = [
    ([5], [-7], 1, -35),                           # 1 cặp bắt buộc
    ([1, 2, 3], [1, 2, 3], 3, 14),                 # k = n = m: ghép theo thứ tự
    ([-1, -1], [5, 5, 5], 2, -10),
    ([10**6] * 3, [-10**6] * 3, 3, -3 * 10**12),
    ([0, 0], [9, -9], 1, 0),
]
for a, b, k, exp in cases:
    r = run_all(a, b, k)
    assert all(v == exp for v in r), (a, b, k, r)
print("Ca biên: OK")

Ca biên: OK


In [16]:
# 12.3 phản ví dụ: khởi tạo mọi ô bằng 0 thay vì -∞ (cho phép "chọn ít cặp hơn")
def wrong_zero_init(a, b, k):
    n, m = len(a), len(b)
    f = [[[0] * (k + 1) for _ in range(m + 1)] for _ in range(n + 1)]       # SAI
    for i in range(1, n + 1):
        for j in range(1, m + 1):
            for t in range(1, k + 1):
                f[i][j][t] = max(f[i - 1][j][t], f[i][j - 1][t], f[i - 1][j - 1][t - 1] + a[i - 1] * b[j - 1])
    return f[n][m][k]

print("sai:", wrong_zero_init([-3, -2], [1, 2], 2), "| đúng:", Solution().maxScore([-3, -2], [1, 2], 2))

sai: 0 | đúng: -7


In [17]:
# 12.4 đối chiếu ngẫu nhiên
rng = random.Random(3836)
for _ in range(300):
    n, m = rng.randint(1, 6), rng.randint(1, 6); k = rng.randint(1, min(n, m))
    a = [rng.randint(-5, 5) for _ in range(n)]; b = [rng.randint(-5, 5) for _ in range(m)]
    r = run_all(a, b, k)
    assert len(set(r)) == 1, (a, b, k, r)
print("300 test nhỏ (có vét cạn): OK")

300 test nhỏ (có vét cạn): OK


In [18]:
# 12.5 test lớn nhất n = m = k = 100 và n = m = 100, k = 50
rng = random.Random(9)
a = [rng.randint(-10**6, 10**6) for _ in range(100)]; b = [rng.randint(-10**6, 10**6) for _ in range(100)]
for k in (50, 100):
    t0 = time.time(); x = Solution().maxScore(a, b, k); t1 = time.time()
    print(f"k={k}: {x}, khớp Cách 4: {x == last_pair(a, b, k)}, thời gian {t1 - t0:.2f}s")

k=50: 29864235178514, khớp Cách 4: True, thời gian 0.06s


k=100: 2392192743555, khớp Cách 4: True, thời gian 0.14s


## 13. Các lỗi người mới hay gặp

### Lỗi 1 — Khởi tạo bằng `0` thay vì `−∞`

Đã thấy ở 12.3: bảng toàn 0 cho phép "dừng ở ít cặp hơn" với điểm 0, nên ví dụ 3 ra `0` thay vì `−7`. **Sửa:** chỉ tầng `t = 0` bằng 0, còn lại `−∞`.

### Lỗi 2 — Tham lam ghép các tích lớn nhất

Chọn `k` tích lớn nhất rồi kiểm tra thứ tự là sai: tích lớn nhất có thể "chặn" các cặp khác.

In [19]:
def greedy_products(a, b, k):
    cand = sorted(((a[i] * b[j], i, j) for i in range(len(a)) for j in range(len(b))), reverse=True)
    chosen = []
    for v, i, j in cand:
        if all((i < i2 and j < j2) or (i > i2 and j > j2) for _, i2, j2 in chosen):
            chosen.append((v, i, j))
        if len(chosen) == k: break
    return sum(v for v, _, _ in chosen) if len(chosen) == k else None

a, b, k = [1, 3, 2], [4, 5, 1], 2
print("tham lam:", greedy_products(a, b, k), "| đúng:", Solution().maxScore(a, b, k))

tham lam: 19 | đúng: 22


**Vì sao sai:** tích lớn nhất `3·5 = 15` dùng `(1, 1)`; các cặp còn tương thích chỉ là `(0, 0)` = 4 hoặc `(2, 2)` = 2 ⇒ tối đa `19` < `22`.

### Lỗi 3 — Quên ràng buộc chỉ số tăng trên **cả hai** dãy

Nếu chỉ yêu cầu `i` tăng (cho phép `j` tùy ý), mỗi phần tử `nums1` được ghép với phần tử tốt nhất của `nums2` ⇒ điểm cao sai.

In [20]:
def wrong_only_i(a, b, k):
    best_per_i = sorted((max(x * y for y in b) for x in a), reverse=True)
    return sum(best_per_i[:k])

print("sai:", wrong_only_i([1, 3, 2], [4, 5, 1], 2), "| đúng:", Solution().maxScore([1, 3, 2], [4, 5, 1], 2))

sai: 25 | đúng: 22


### Lỗi 4 — Dùng một bảng 2D duy nhất cập nhật "tại chỗ" cho mọi `t`

Nếu ghi đè tầng `t − 1` trong khi vẫn cần đọc `prev[i-1][j-1]` của tầng cũ, giá trị bị lẫn tầng (một phần tử dùng hai lần).

In [21]:
def wrong_inplace(a, b, k):
    n, m = len(a), len(b)
    f = [[0] * (m + 1) for _ in range(n + 1)]
    for t in range(1, k + 1):
        for i in range(1, n + 1):
            for j in range(1, m + 1):
                f[i][j] = max(f[i - 1][j] if i > 1 or t == 0 else NEG, f[i][j - 1] if j > 1 else NEG,
                              f[i - 1][j - 1] + a[i - 1] * b[j - 1])            # SAI: f[i-1][j-1] đã là tầng t
    return f[n][m]

rng = random.Random(8)
for _ in range(5000):
    n, m = rng.randint(1, 5), rng.randint(1, 5); k = rng.randint(1, min(n, m))
    a = [rng.randint(-5, 5) for _ in range(n)]; b = [rng.randint(-5, 5) for _ in range(m)]
    if wrong_inplace(a, b, k) != Solution().maxScore(a, b, k):
        print("phản ví dụ:", a, b, k, "| sai:", wrong_inplace(a, b, k), "| đúng:", Solution().maxScore(a, b, k)); break

phản ví dụ: [1, 5] [-5, 2, 2, 2, 1] 1 | sai: 12 | đúng: 10


## 14. Mẹo làm bài nhanh & nhận dạng pattern

- **"Chọn cặp từ hai dãy với chỉ số tăng trên cả hai"** ⇒ DP kiểu LCS `f[i][j]` (LC 1458 *Max Dot Product of Two Subsequences*, LC 1035 *Uncrossed Lines*).
- **"Đúng `k`"** ⇒ thêm chiều đếm + `−∞` cho trạng thái không hợp lệ.
- Giá trị **âm** ⇒ tuyệt đối không khởi tạo 0 cho trạng thái "đã chọn ≥ 1".
- Chiều đếm chỉ phụ thuộc tầng trước ⇒ cuộn mảng theo tầng để tiết kiệm bộ nhớ.

## 15. Biến thể của bài

1. **Không bắt buộc đúng `k`** (tối đa `k`, ít nhất 1): đáp án `max_{1 ≤ t ≤ k} f[n][m][t]`.
2. **Không giới hạn số cặp** (ít nhất 1 cặp): LC 1458 — bỏ chiều `t`.
3. **Cặp liên tiếp phải cách nhau ít nhất `d` chỉ số**: dùng Cách 4 với max tiền tố tại `(i−d, j−d)`.
4. **Điểm là `|a − b|`** hoặc hàm bất kỳ `w(i, j)`: thay tích bằng `w(i, j)`.

In [22]:
# Biến thể 2: LC 1458 (ít nhất 1 cặp, số cặp tùy ý)
def max_dot_product(a, b):
    n, m = len(a), len(b)
    f = [[NEG] * (m + 1) for _ in range(n + 1)]
    for i in range(1, n + 1):
        for j in range(1, m + 1):
            p = a[i - 1] * b[j - 1]
            f[i][j] = max(f[i - 1][j], f[i][j - 1], p, p + max(f[i - 1][j - 1], 0) if f[i - 1][j - 1] != NEG else p)
    return f[n][m]

print(max_dot_product([2, 1, -2, 5], [3, 0, -6]), "(LC 1458 ví dụ 1 cho 18)")
assert max_dot_product([2, 1, -2, 5], [3, 0, -6]) == max(Solution().maxScore([2, 1, -2, 5], [3, 0, -6], t) for t in range(1, 4))

18 (LC 1458 ví dụ 1 cho 18)


## 16. So sánh các cách làm

| Cách | Ý tưởng | Time | Space | Khi nào dùng |
|---|---|---|---|---|
| 1. Vét cạn | Mọi bộ chỉ số | tổ hợp | `O(k)` | Kiểm thử |
| 2. Memo | `dfs(i, j, t)` | `O(nmk)` | `O(nmk)` | Dễ viết từ định nghĩa |
| 3. DP theo tầng | Hai bảng `(n+1)×(m+1)` | `O(nmk)` | `O(nm)` | **Lời giải chuẩn** |
| 4. Cặp cuối + max tiền tố | `h_t = a·b + prefmax(h_{t−1})` | `O(nmk)` | `O(nm)` | Biến thể có khoảng cách |

## 17. 5 bài tập vận dụng (từ dễ đến khó)

**Bài 1 (đoán output).** `maxScore([2, -1], [-3, 4], 1)` = ?

**Bài 2 (tính tay).** `maxScore([1, -2, 3], [-1, 2], 2)` — liệt kê các cách chọn.

**Bài 3 (điền TODO).** Viết `max_score_at_most(a, b, k)` (tối đa `k` cặp, ít nhất 1).

**Bài 4 (tự viết lại).** Viết lại Cách 3 với **một** mảng 2D và biến tạm lưu `prev[i−1][j−1]` (cập nhật đúng tầng).

**Bài 5 (mini challenge).** Ràng buộc thêm: hai cặp liên tiếp phải thỏa `i_{s+1} − i_s ≥ 2` và `j_{s+1} − j_s ≥ 2`. Viết lời giải và kiểm chứng.

In [23]:
# Bài 3 — TODO (không gọi hàm này khi chạy notebook)
def max_score_at_most(a, b, k):
    # TODO: chạy DP theo tầng, lấy max f_t[n][m] với t = 1..k
    pass

In [24]:
# Bài 5 — TODO (không gọi hàm này khi chạy notebook)
def max_score_gap2(a, b, k):
    # TODO: h_t[i][j] = a[i]*b[j] + prefmax(h_{t-1})[i-2][j-2]
    pass

## 18. Gợi ý 3 mức cho bài tập

**Bài 1.** (1) Một cặp bất kỳ. (2) Tích lớn nhất: `2·4 = 8`, `(−1)(−3) = 3`. (3) Đáp án `8`.

**Bài 2.** (1) Hai cặp phải dùng `b[0] = −1` và `b[1] = 2` theo thứ tự. (2) Cặp `(i₁, 0)`, `(i₂, 1)` với `i₁ < i₂`: `(1,·)`: `(−2)(−1) + 3·2 = 8`; `(0,·)`: `−1 + 3·2 = 5`, `−1 + (−2)·2 = −5`. (3) Đáp án `8`.

**Bài 3.** (1) Lưu đáp án mỗi tầng. (2) Lấy max. (3) Đối chiếu vét cạn với mọi `t ≤ k`.

**Bài 4.** (1) Duyệt `i` tăng, `j` tăng. (2) Trước khi ghi `f[i][j]`, giữ `diag = f_old[i−1][j−1]` từ hàng trước (lưu bản sao hàng trước). (3) So với `Solution`.

**Bài 5.** (1) Dùng Cách 4. (2) Max tiền tố tại `(i−2, j−2)`. (3) Vét cạn lọc các bộ chỉ số có khoảng cách ≥ 2.

## 19. Đáp án tham khảo cho bài tập

In [25]:
# Bài 1, 2
assert Solution().maxScore([2, -1], [-3, 4], 1) == 8
assert Solution().maxScore([1, -2, 3], [-1, 2], 2) == 8 == brute([1, -2, 3], [-1, 2], 2)
print("Bài 1, 2 OK")

Bài 1, 2 OK


In [26]:
# Bài 3
def max_score_at_most_sol(a, b, k):
    n, m = len(a), len(b)
    prev = [[0] * (m + 1) for _ in range(n + 1)]; best = NEG
    for t in range(1, k + 1):
        cur = [[NEG] * (m + 1) for _ in range(n + 1)]
        for i in range(1, n + 1):
            for j in range(1, m + 1):
                cur[i][j] = max(cur[i - 1][j], cur[i][j - 1], prev[i - 1][j - 1] + a[i - 1] * b[j - 1])
        best = max(best, cur[n][m]); prev = cur
    return best

rng = random.Random(3)
for _ in range(200):
    n, m = rng.randint(1, 5), rng.randint(1, 5); k = rng.randint(1, min(n, m))
    a = [rng.randint(-5, 5) for _ in range(n)]; b = [rng.randint(-5, 5) for _ in range(m)]
    assert max_score_at_most_sol(a, b, k) == max(brute(a, b, t) for t in range(1, k + 1))
print("Bài 3 OK")

Bài 3 OK


In [27]:
# Bài 4: một mảng 2D + bản sao hàng trước của tầng cũ
def one_table(a, b, k):
    n, m = len(a), len(b)
    f = [[0] * (m + 1) for _ in range(n + 1)]                 # tầng 0
    for t in range(1, k + 1):
        old_prev_row = f[0][:]                                  # hàng i-1 của tầng t-1
        f[0] = [NEG] * (m + 1)
        for i in range(1, n + 1):
            old_row = f[i][:]                                   # hàng i của tầng t-1 (để dùng cho i+1)
            f[i][0] = NEG
            for j in range(1, m + 1):
                f[i][j] = max(f[i - 1][j], f[i][j - 1], old_prev_row[j - 1] + a[i - 1] * b[j - 1])
            old_prev_row = old_row
    return f[n][m]

rng = random.Random(4)
for _ in range(200):
    n, m = rng.randint(1, 7), rng.randint(1, 7); k = rng.randint(1, min(n, m))
    a = [rng.randint(-9, 9) for _ in range(n)]; b = [rng.randint(-9, 9) for _ in range(m)]
    assert one_table(a, b, k) == Solution().maxScore(a, b, k)
print("Bài 4 OK")

Bài 4 OK


In [28]:
# Bài 5: khoảng cách >= 2 giữa hai cặp liên tiếp
def gap2_sol(a, b, k):
    n, m = len(a), len(b)
    h = [[a[i] * b[j] for j in range(m)] for i in range(n)]
    for t in range(2, k + 1):
        M = [[NEG] * m for _ in range(n)]
        for i in range(n):
            for j in range(m):
                M[i][j] = max(h[i][j], M[i - 1][j] if i else NEG, M[i][j - 1] if j else NEG)
        h = [[a[i] * b[j] + (M[i - 2][j - 2] if i >= 2 and j >= 2 else NEG) for j in range(m)] for i in range(n)]
    return max(max(r) for r in h)

def brute_gap2(a, b, k):
    best = NEG
    for I in combinations(range(len(a)), k):
        if any(I[s + 1] - I[s] < 2 for s in range(k - 1)): continue
        for J in combinations(range(len(b)), k):
            if any(J[s + 1] - J[s] < 2 for s in range(k - 1)): continue
            best = max(best, sum(a[i] * b[j] for i, j in zip(I, J)))
    return best

rng = random.Random(5)
for _ in range(200):
    n, m = rng.randint(1, 7), rng.randint(1, 7); k = rng.randint(1, max(1, min((n + 1) // 2, (m + 1) // 2)))
    a = [rng.randint(-5, 5) for _ in range(n)]; b = [rng.randint(-5, 5) for _ in range(m)]
    assert gap2_sol(a, b, k) == brute_gap2(a, b, k)
print("Bài 5 OK")

Bài 5 OK


## 20. Template nhớ nhanh + Checklist tự kiểm tra trước khi submit

```python
prev = [[0]*(m+1) for _ in range(n+1)]          # t = 0
for t in 1..k:
    cur = [[-inf]*(m+1) for _ in range(n+1)]
    for i in 1..n:
        for j in 1..m:
            cur[i][j] = max(cur[i-1][j], cur[i][j-1],
                            prev[i-1][j-1] + a[i-1]*b[j-1])
    prev = cur
return prev[n][m]
```

**Checklist:**

- [ ] Tầng 0 bằng 0, các tầng `t ≥ 1` khởi tạo `−∞` (kể cả hàng/cột 0)?
- [ ] Ba chuyển trạng thái: bỏ `nums1`, bỏ `nums2`, ghép?
- [ ] Ghép dùng giá trị của **tầng trước** `prev[i−1][j−1]`?
- [ ] Không tham lam theo tích lớn nhất?
- [ ] Giá trị tới `10^{14}` (an toàn trong Python; ngôn ngữ khác dùng `long long`)?